# TissueMC: why do R3's warps wait so long?

R3 raised the active lanes per warp from 4.74 to 21.61 of 32 but ran about 6% slower than R1, because each warp waited about 98 cycles between issued instructions instead of 20. The counters collected so far do not say why.

This notebook collects **every** Nsight Compute counter for R1 and R3 on the same workload and prints the ones that bear on the question side by side: the warp stall reasons, the instruction counts, the local-memory traffic and the launch and occupancy figures.

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Nothing else to record.

**Step 1 is a short check** of the whole path on a tiny run. The long steps start only if it passes; otherwise they print "SKIPPED", so *Run all* cannot waste time on a broken run. If step 1 fails, **paste its output**. Raw counters are kept next to each result (`results/*.ncu.csv`) and reused if you run a step again. The full profiles replay the kernel about forty times each: roughly 8 minutes for R1 and 3 for R3. The notebook ends with **one digest** for you to paste back. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Safe to run twice: reuse the clone if it is already there instead of nesting a second one.
!if [ -d /content/tissuemc/.git ]; then git -C /content/tissuemc pull -q; else git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git /content/tissuemc; fi
%cd /content/tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

In [ ]:
# 1. Quick check of the whole path on a tiny run: collect, parse, digest. The long steps below only
# run if this passes (it leaves results/.smoke_ok).
!rm -f results/.smoke_ok
!tissuemc-profile --rung 1 --full --reuse-csv --photons 20000 --binary build/tissuemc --output results/smoke-full.json && tissuemc-profile --digest results/smoke-full.json results/smoke-full.json && touch results/.smoke_ok

In [ ]:
# 2. Every counter for R1 (500,000 photons). About 8 minutes.
!if [ -f results/.smoke_ok ]; then tissuemc-profile --rung 1 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r1-full.json; else echo "SKIPPED: step 1 did not pass, so this long run was not started"; fi

In [ ]:
# 3. Every counter for R3, the same workload. About 3 minutes.
!if [ -f results/.smoke_ok ]; then tissuemc-profile --rung 3 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r3-full.json; else echo "SKIPPED: step 1 did not pass, so this long run was not started"; fi

In [ ]:
# 4. The digest: A is R1, B is R3. Paste this cell's output back.
!if [ -f results/gpu-r1-full.json ] && [ -f results/gpu-r3-full.json ]; then tissuemc-profile --digest results/gpu-r1-full.json results/gpu-r3-full.json; else echo "SKIPPED: a profile above did not finish"; fi